In [ ]:
from pathlib import Path
import sys
sys.path.append(str(Path.cwd().parent))

import joblib
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import IPython.display as ipd
import soundfile as sf

from art_tts.paths import DATA_DIR

import art_tts.utils_ema.ema_dataset

In [ ]:
def get_gt_audio(filestem, wav_dir):
    original_path = wav_dir / f"{filestem}.wav"
    audio, sr = sf.read(original_path)
    ipd.display(ipd.Audio(audio, rate=sr))

def get_sparc_audio(filestem, hifigan_dir, vocoder="sparc_multi"):
    audio_path = hifigan_dir / "sparc" / vocoder / f"{filestem}.wav"
    audio, sr = sf.read(audio_path)
    ipd.display(ipd.Audio(audio, rate=sr))

def get_model_audios(filestem, version, ckpt_name, hifigan_dir, show_enc=False, suffix="", vocoder="sparc_multi"):
    print(f"\t Enc/dec from {version} {ckpt_name}")
    if show_enc:
        audio_path_enc = hifigan_dir / version / ckpt_name / vocoder / f"{filestem}_encoder{suffix}.wav"
        audio_enc, sr_enc = sf.read(audio_path_enc)
        ipd.display(ipd.Audio(audio_enc, rate=sr_enc))
    audio_path_dec = hifigan_dir / version / ckpt_name / vocoder / f"{filestem}_decoder{suffix}.wav"
    audio_dec, sr_dec = sf.read(audio_path_dec)
    ipd.display(ipd.Audio(audio_dec, rate=sr_dec))

In [ ]:
import random

def get_best_worst_random_samples(analysis_df, overall=False):
    
    grouped = analysis_df.groupby("lang")

    # Initialize a dictionary to store results
    results = {}

    # Iterate through each language group
    for lang, group in grouped:
        if overall:
            group["overall"] = group[["pcc_ema", "pcc_pitch", "pcc_loudness"]].mean(axis=1)
            best_sample = group.loc[group["overall"].idxmax()]  # Best score
            worst_sample = group.loc[group["overall"].idxmin()]  # Worst score
            random_sample = group.sample(n=1).iloc[0]  # Random sample

        else:
            best_sample = group.loc[group["pcc_ema"].idxmax()]  # Best score
            worst_sample = group.loc[group["pcc_ema"].idxmin()]  # Worst score
            random_sample = group.sample(n=1).iloc[0]  # Random sample
        
        results[lang] = {
            "best": best_sample,
            "worst": worst_sample,
            "random": random_sample
        }

    ## Display results
    #for lang, samples in results.items():
    #    print(f"Language: {lang}")
    #    print("Best sample:", samples["best"])
    #    print("Worst sample:", samples["worst"])
    #    print("Random sample:", samples["random"])
    #    print()
    return results

# v6

In [ ]:
from art_tts.voxcommunis.io import read_manifest

languages = ["it", "sw", "zh-CN"]
languages = ["it", "zh-CN"]
languages = ["it", "fr", "zh-CN"]
vc_dir = DATA_DIR / "VoxCommunis"
split = "dev-1h"

version = "v6"
epoch = 5000
gradname = f"grad_{epoch}"
analysis_df = pd.read_csv(vc_dir / split / "analysis" / f"quanti_art_comp_{version}_{gradname}.csv")
analysis_df["lang"] = analysis_df["sample_id"].apply(lambda x: x.split("_")[2])
results = get_best_worst_random_samples(analysis_df, overall=True)

In [ ]:
#cross lingual
dataset = "VoxCommunis"
version_1 = "v6"
ckpt_name_1 = "grad_5000"
version_2 = "v6_zhCN"
ckpt_name_2 = "grad_5000"

hifigan_dir = DATA_DIR / dataset / split / "hifigan_pred"
#wav_dir = DATA_DIR / dataset / "wavs"

for lang in languages:
    print(f"Language: {lang}")
    for score_type, v in results[lang].items():
        print(f"Type : {score_type}, sample_id : {v['sample_id']}, pcc_ema : {v['pcc_ema']:.3f}, pcc_pitch : {v['pcc_pitch']:.3f}, \
pcc_loudness : {v['pcc_loudness']:.3f}")
        filestem = v["sample_id"]
        #get_gt_audio(filestem, wav_dir)
        get_sparc_audio(filestem, hifigan_dir)
        get_model_audios(filestem, version_1, ckpt_name_1, hifigan_dir)
        get_model_audios(filestem, version_2, ckpt_name_2, hifigan_dir)

In [ ]:
#cross epoch
dataset = "VoxCommunis"
version_1 = "v6"
ckpt_name_1 = "grad_2000"
version_2 = "v6"
ckpt_name_2 = "grad_5000"

hifigan_dir = DATA_DIR / dataset / split / "hifigan_pred"
#wav_dir = DATA_DIR / dataset / "wavs"

for lang in languages:
    print(f"Language: {lang}")
    for score_type, v in results[lang].items():
        print(f"Type : {score_type}, sample_id : {v['sample_id']}, pcc_ema : {v['pcc_ema']:.3f}, pcc_pitch : {v['pcc_pitch']:.3f}, \
pcc_loudness : {v['pcc_loudness']:.3f}")
        filestem = v["sample_id"]
        #get_gt_audio(filestem, wav_dir)
        #get_sparc_audio(filestem, hifigan_dir)
        get_model_audios(filestem, version_1, ckpt_name_1, hifigan_dir)
        get_model_audios(filestem, version_2, ckpt_name_2, hifigan_dir)

# v6_zhCN

In [ ]:
from art_tts.voxcommunis.io import read_manifest

languages = ["it", "sw", "zh-CN"]
languages = ["it", "zh-CN"]
languages = ["it", "fr", "zh-CN"]
vc_dir = DATA_DIR / "VoxCommunis"
split = "dev-1h"

version = "v6_zhCN"
epoch = 5000
gradname = f"grad_{epoch}"
analysis_df = pd.read_csv(vc_dir / split / "analysis" / f"quanti_art_comp_{version}_{gradname}.csv")
analysis_df["lang"] = analysis_df["sample_id"].apply(lambda x: x.split("_")[2])
results = get_best_worst_random_samples(analysis_df)

In [ ]:
#cross lingual comparison
dataset = "VoxCommunis"
version_1 = "v6"
ckpt_name_1 = "grad_5000"
version_2 = "v6_zhCN"
ckpt_name_2 = "grad_5000"

hifigan_dir = DATA_DIR / dataset / split / "hifigan_pred"
#wav_dir = DATA_DIR / dataset / "wavs"

for lang in languages:
    print(f"Language: {lang}")
    for score_type, v in results[lang].items():
        print(f"Type : {score_type}, sample_id : {v['sample_id']}, pcc_ema : {v['pcc_ema']:.3f}, pcc_pitch : {v['pcc_pitch']:.3f}, \
pcc_loudness : {v['pcc_loudness']:.3f}")
        filestem = v["sample_id"]
        #get_gt_audio(filestem, wav_dir)
        get_sparc_audio(filestem, hifigan_dir)
        get_model_audios(filestem, version_1, ckpt_name_1, hifigan_dir)
        get_model_audios(filestem, version_2, ckpt_name_2, hifigan_dir)

In [ ]:
#cross epoch comparison
dataset = "VoxCommunis"
version_1 = "v6"
ckpt_name_1 = "grad_1000"
version_2 = "v6_zhCN"
ckpt_name_2 = "grad_5000"

hifigan_dir = DATA_DIR / dataset / split / "hifigan_pred"
#wav_dir = DATA_DIR / dataset / "wavs"

for lang in languages:
    print(f"Language: {lang}")
    for score_type, v in results[lang].items():
        print(f"Type : {score_type}, sample_id : {v['sample_id']}, pcc_ema : {v['pcc_ema']:.3f}, pcc_pitch : {v['pcc_pitch']:.3f}, \
pcc_loudness : {v['pcc_loudness']:.3f}")
        filestem = v["sample_id"]
        #get_gt_audio(filestem, wav_dir)
        #get_sparc_audio(filestem, hifigan_dir)
        get_model_audios(filestem, version_1, ckpt_name_1, hifigan_dir)
        get_model_audios(filestem, version_2, ckpt_name_2, hifigan_dir)

# msml1h

In [ ]:
from art_tts.voxcommunis.io import read_manifest

languages = ["it", "fr", "zh-CN"]
vc_dir = DATA_DIR / "VoxCommunis"
split = "dev-1h"

version = "msml1h"
epoch = 1000
gradname = f"grad_{epoch}"
analysis_df = pd.read_csv(vc_dir / split / "analysis" / f"quanti_art_comp_{version}_{gradname}.csv")
analysis_df["lang"] = analysis_df["sample_id"].apply(lambda x: x.split("_")[2])
results = get_best_worst_random_samples(analysis_df)

In [ ]:
dataset = "VoxCommunis"

hifigan_dir = DATA_DIR / dataset / split / "hifigan_pred"
#wav_dir = DATA_DIR / dataset / "wavs"
versions = ["v6", "v6_zhCN", "msml1h", "msml1h"]
ckpt_names = ["grad_5000", "grad_5000", "grad_2000", "grad_4000"]
vocoders = ["sparc_multi", "sparc_multi", "sparc_multi", "sparc_multi"]

for lang in languages:
    print(f"Language: {lang}")
    for score_type, v in results[lang].items():
        print(f"Type : {score_type}, sample_id : {v['sample_id']}, pcc_ema : {v['pcc_ema']:.3f}, pcc_pitch : {v['pcc_pitch']:.3f}, \
pcc_loudness : {v['pcc_loudness']:.3f}")
        filestem = v["sample_id"]
        #get_gt_audio(filestem, wav_dir)
        get_sparc_audio(filestem, hifigan_dir)
        for version, ckpt_name, vocoder in zip(versions, ckpt_names, vocoders):
            get_model_audios(filestem, version, ckpt_name, hifigan_dir, vocoder=vocoder)

# custom datasets

In [ ]:
def get_best_worst_random_samples_custom(analysis_df, spk_parse_idx=0, overall=False):
    analysis_df["spk"] = analysis_df["sample_id"].apply(lambda x: x.split("_")[spk_parse_idx])
    grouped = analysis_df.groupby("spk")

    # Initialize a dictionary to store results
    results = {}

    # Iterate through each language group
    for spk, group in grouped:
        if overall:
            group["overall"] = group[["pcc_sparc_dec_ema", "pcc_sparc_dec_pitch", "pcc_sparc_dec_loudness"]].mean(axis=1)
            best_sample = group.loc[group["overall"].idxmax()]  # Best score
            worst_sample = group.loc[group["overall"].idxmin()]  # Worst score
            random_sample = group.sample(n=1).iloc[0]  # Random sample

        else:
            best_sample = group.loc[group["pcc_sparc_dec_ema"].idxmax()]  # Best score
            worst_sample = group.loc[group["pcc_sparc_dec_ema"].idxmin()]  # Worst score
            random_sample = group.sample(n=1).iloc[0]  # Random sample
        
        results[spk] = {
            "best": best_sample,
            "worst": worst_sample,
            "random": random_sample
        }
    return results

## MSPKA

In [ ]:
from art_tts.voxcommunis.io import read_manifest

vc_dir = DATA_DIR / "VoxCommunis"
split = "MSPKA_EMA_ita"
spk_parse_idx = 0

version = "msml1h"
epoch = 4000
gradname = f"grad_{epoch}"
analysis_df = pd.read_csv(vc_dir / split / "analysis" / f"quanti_gt_art_comp_{version}_{gradname}.csv")
results = get_best_worst_random_samples_custom(analysis_df, spk_parse_idx=spk_parse_idx, overall=True)

In [ ]:
#cross epoch
dataset = "VoxCommunis"
versions = ["v6", "v6_zhCN", "msml1h", "msml1h"]
ckpt_names = ["grad_3000", "grad_3000", "grad_2000", "grad_4000"]
vocoders = ["sparc_multi", "sparc_multi", "sparc_multi", "sparc_multi"]

hifigan_dir = DATA_DIR / dataset / split / "hifigan_pred"
#wav_dir = DATA_DIR / dataset / "wavs"

for spk in results.keys():
    for score_type, v in results[spk].items():
        print(f"Type : {score_type}, sample_id : {v['sample_id']}, pcc_sparc_dec_ema : {v['pcc_sparc_dec_ema']:.3f}, pcc_sparc_dec_pitch : {v['pcc_sparc_dec_pitch']:.3f}, \
pcc_sparc_dec_loudness : {v['pcc_sparc_dec_loudness']:.3f}")
        filestem = v["sample_id"]
        #get_gt_audio(filestem, wav_dir)
        get_sparc_audio(filestem, hifigan_dir)
        for version, ckpt_name, vocoder in zip(versions, ckpt_names, vocoders):
            get_model_audios(filestem, version, ckpt_name, hifigan_dir, show_enc=False, vocoder=vocoder)

Average pretty good and consistently across the different models, much better than voxcom samples, sounds a bit worse on olm though. It validates the hypothesis that lower quality audio affect speaker embedding for articulatory generation and maybe HiFi-GAN generation as well. Especially given that the articulatory scores are poor (around 0.2 0.3 compared to both gt and sparc) it doesn't mean that it's due to being closer to sparc articulatory distribution on which the HiFi-GAN was tuned on. Mystery how meaningfully content is kept despite low PCC correlation

## Mocha

In [ ]:
from art_tts.voxcommunis.io import read_manifest

vc_dir = DATA_DIR / "VoxCommunis"
split = "mocha_timit"
spk_parse_idx = 0

version = "msml1h"
epoch = 4000
gradname = f"grad_{epoch}"
analysis_df = pd.read_csv(vc_dir / split / "analysis" / f"quanti_gt_art_comp_{version}_{gradname}.csv")
results = get_best_worst_random_samples_custom(analysis_df, spk_parse_idx=spk_parse_idx, overall=True)

In [ ]:
#cross epoch
dataset = "VoxCommunis"
versions = ["v6", "v6_zhCN", "msml1h", "msml1h"]
ckpt_names = ["grad_3000", "grad_3000", "grad_2000", "grad_4000"]
vocoders = ["sparc_multi", "sparc_multi", "sparc_multi", "sparc_multi"]

hifigan_dir = DATA_DIR / dataset / split / "hifigan_pred"
#wav_dir = DATA_DIR / dataset / "wavs"

for spk in results.keys():
    for score_type, v in results[spk].items():
        print(f"Type : {score_type}, sample_id : {v['sample_id']}, pcc_sparc_dec_ema : {v['pcc_sparc_dec_ema']:.3f}, pcc_sparc_dec_pitch : {v['pcc_sparc_dec_pitch']:.3f}, \
pcc_sparc_dec_loudness : {v['pcc_sparc_dec_loudness']:.3f}")
        filestem = v["sample_id"]
        #get_gt_audio(filestem, wav_dir)
        get_sparc_audio(filestem, hifigan_dir)
        for version, ckpt_name, vocoder in zip(versions, ckpt_names, vocoders):
            get_model_audios(filestem, version, ckpt_name, hifigan_dir, show_enc=False, vocoder=vocoder)

Not very good maybe sparc_multi hifigan not suited for english (even sparc not good actually little intelligibility). Some are ok most are bad 

## MNGU0

In [ ]:
from art_tts.voxcommunis.io import read_manifest

vc_dir = DATA_DIR / "VoxCommunis"
split = "MNGU0"
spk_parse_idx = 1

version = "msml1h"
epoch = 4000
gradname = f"grad_{epoch}"
analysis_df = pd.read_csv(vc_dir / split / "analysis" / f"quanti_gt_art_comp_{version}_{gradname}.csv")
results = get_best_worst_random_samples_custom(analysis_df, spk_parse_idx=spk_parse_idx, overall=True)

In [ ]:
#cross epoch
dataset = "VoxCommunis"
versions = ["v6", "v6_zhCN", "msml1h", "msml1h"]
ckpt_names = ["grad_3000", "grad_3000", "grad_2000", "grad_4000"]
vocoders = ["sparc_multi", "sparc_multi", "sparc_multi", "sparc_multi"]

hifigan_dir = DATA_DIR / dataset / split / "hifigan_pred"
#wav_dir = DATA_DIR / dataset / "wavs"

for spk in results.keys():
    for score_type, v in results[spk].items():
        print(f"Type : {score_type}, sample_id : {v['sample_id']}, pcc_sparc_dec_ema : {v['pcc_sparc_dec_ema']:.3f}, pcc_sparc_dec_pitch : {v['pcc_sparc_dec_pitch']:.3f}, \
pcc_sparc_dec_loudness : {v['pcc_sparc_dec_loudness']:.3f}")
        filestem = v["sample_id"]
        #get_gt_audio(filestem, wav_dir)
        get_sparc_audio(filestem, hifigan_dir)
        for version, ckpt_name, vocoder in zip(versions, ckpt_names, vocoders):
            get_model_audios(filestem, version, ckpt_name, hifigan_dir, show_enc=False, vocoder=vocoder)

Better than mocha relatively intelligible for every model despite some being gibberish, not an issue of sparc_multi hifigan on english. Reinforces our conviction that its due to the spk emb. Still not very good

## pb2007

In [ ]:
from art_tts.voxcommunis.io import read_manifest

vc_dir = DATA_DIR / "VoxCommunis"
split = "pb2007"
spk_parse_idx = 0

version = "msml1h"
epoch = 4000
gradname = f"grad_{epoch}"
analysis_df = pd.read_csv(vc_dir / split / "analysis" / f"quanti_gt_art_comp_{version}_{gradname}.csv")
#only keep analysis_df for samples being sentences (rank above 1030)
analysis_df["rank"] = analysis_df["sample_id"].apply(lambda x: x.split("_")[1])
analysis_df = analysis_df[analysis_df["rank"].astype(int) > 1030]
analysis_df = analysis_df[analysis_df["rank"].astype(int) != 1083]
analysis_df = analysis_df[analysis_df["rank"].astype(int) != 1081]
results = get_best_worst_random_samples_custom(analysis_df, spk_parse_idx=spk_parse_idx, overall=True)

In [ ]:
#cross epoch
dataset = "VoxCommunis"
versions = ["v6", "v6_zhCN", "msml1h"]
ckpt_names = ["grad_3000", "grad_3000", "grad_4000"]
vocoders = ["sparc_multi", "sparc_multi", "sparc_multi"]

hifigan_dir = DATA_DIR / dataset / split / "hifigan_pred"
#wav_dir = DATA_DIR / dataset / "wavs"

for spk in results.keys():
    for score_type, v in results[spk].items():
        print(f"Type : {score_type}, sample_id : {v['sample_id']}, pcc_sparc_dec_ema : {v['pcc_sparc_dec_ema']:.3f}, pcc_sparc_dec_pitch : {v['pcc_sparc_dec_pitch']:.3f}, \
pcc_sparc_dec_loudness : {v['pcc_sparc_dec_loudness']:.3f}")
        filestem = v["sample_id"]
        #get_gt_audio(filestem, wav_dir)
        get_sparc_audio(filestem, hifigan_dir)
        for version, ckpt_name, vocoder in zip(versions, ckpt_names, vocoders):
            get_model_audios(filestem, version, ckpt_name, hifigan_dir, show_enc=False, vocoder=vocoder)

Not bad, once again speech quality and articulatory scores are uncorrelated